<a href="https://colab.research.google.com/github/shatrill/data_analytics/blob/main/session_7_eda_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 7: Exploratory Data Analysis (EDA) in Python

**Course:** Data Analytics Foundation (CompTIA Data+) | **CompTIA Domain:** 2.0 Data Mining | 3.0 Data Analysis | 4.0 Data Visualization

> Open this notebook in **Google Colab**. EDA is detective work: before you model or report anything, you interrogate the data to learn its shape, its center, its spread, its outliers, and how variables relate to one another.

## 🗺️ Your 45-Minute Lab Map
| Part | What you will do | Approx. time |
|---|---|---|
| Part 1 | Univariate vs. bivariate analysis (concepts) | 3 min |
| Part 2 | Univariate statistics on the tips data: mean, median, mode, variance, std, IQR | 8 min |
| Part 3 | Visual EDA: histogram, box plot, scatter plot, correlation heatmap | 9 min |
| Part 4 | Guided Example: detect outliers with the 1.5 × IQR rule | 10 min |
| Part 5 | ✏️ Your Turn: Palmer Penguins EDA sprint | 8 min |
| Part 6 | 🧩 Guided Challenge: penguins deep dive (relationships + outliers by group) | 7 min |

**How to use this notebook:** read each text cell, then run the code cell beneath it with **Shift + Enter**.
Cells marked **✏️ Your Turn** have blanks (`___`) for you to fill in. Each one has a hidden solution
you can reveal *after* you have tried it. Errors are normal and part of learning. Read the last line
of the error message, fix it, and run the cell again.

## Part 1: The Levels of EDA

| Level | Looks at | Questions it answers | Typical tools |
|---|---|---|---|
| **Univariate** | **One** variable at a time | What is typical? How spread out? Any weird values? | mean/median/mode, std, IQR, histogram, box plot |
| **Bivariate** | **Two** variables together | Do they move together? Do groups differ? | scatter plot, grouped box plot, correlation, crosstab |
| **Multivariate** | Three or more | How do many variables interact? | heatmaps, color-coded (hue) plots |

**Which chart for which pair?**
- Numeric vs. numeric → **scatter plot** and correlation
- Category vs. numeric → **grouped box plot** or bar chart
- Category vs. category → **crosstab** (contingency table)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")        # clean default look for every chart
tips = sns.load_dataset("tips")         # 244 restaurant bills, built into seaborn
tips.head()

## Part 2: Univariate Statistics

### Measures of center
- **Mean**: the average. It is sensitive to extreme values.
- **Median**: the middle value when sorted. It resists extreme values.
- **Mode**: the most frequent value. It is the only center that works for categories.

In [ ]:
bill = tips["total_bill"]

print("Mean:  ", round(bill.mean(), 2))
print("Median:", round(bill.median(), 2))
print("Mode:  ", bill.mode()[0], "(most common bill amount)")
print("Most common day:", tips["day"].mode()[0])

### Measures of spread
- **Range** = max − min.
- **Variance** = the average *squared* distance from the mean (in squared dollars, which is hard to interpret).
- **Standard deviation** = the square root of variance, back in normal dollars. It describes the "typical distance from the mean."
- **IQR (Interquartile Range)** = Q3 − Q1, the width of the middle 50% of the data.

In [ ]:
q1, q3 = bill.quantile(0.25), bill.quantile(0.75)

print("Range:    ", round(bill.max() - bill.min(), 2))
print("Variance: ", round(bill.var(), 2))
print("Std dev:  ", round(bill.std(), 2))
print("Q1:", q1, "| Q3:", q3, "| IQR:", round(q3 - q1, 2))

**💬 Interpretation:** the mean ($19.79) is **greater than** the median ($17.80). That signals a **right-skewed** distribution: a few large bills pull the mean upward. Let's *see* that.

## Part 3: Visual EDA

### 3A. Histogram: the shape of one variable

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(tips["total_bill"], bins=20, kde=True)
plt.axvline(bill.mean(),   color="red",   linestyle="--", label=f"Mean ${bill.mean():.2f}")
plt.axvline(bill.median(), color="green", linestyle="-",  label=f"Median ${bill.median():.2f}")
plt.title("Distribution of Total Bill")
plt.xlabel("Total bill ($)")
plt.legend()
plt.show()

The long tail stretches **right**, and the mean sits to the right of the median. This is a textbook right skew (a CompTIA favorite!).

### 3B. Box plot: center, spread and outliers at a glance

In [ ]:
plt.figure(figsize=(8, 3))
sns.boxplot(x=tips["total_bill"])
plt.title("Box Plot of Total Bill")
plt.show()

How to read a box plot: the **box** spans Q1 to Q3 (the IQR), the **line** inside is the median, the **whiskers** reach the most extreme points within 1.5 × IQR, and the **dots** beyond them are potential outliers.

### 3C. Scatter plot: two numeric variables (bivariate)

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=tips, x="total_bill", y="tip", hue="time")
plt.title("Tip vs. Total Bill")
plt.show()

### 3D. Correlation heatmap: every numeric pair at once
Pearson's *r* ranges from **−1** (perfect negative) through **0** (no linear relationship) to **+1** (perfect positive). Remember: **correlation does not imply causation.**

In [ ]:
corr = tips.corr(numeric_only=True)

plt.figure(figsize=(5, 4))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Heatmap (tips)")
plt.show()

**💬 Insight:** `total_bill` and `tip` have *r* ≈ 0.68, a strong positive relationship. Bigger bills tend to get bigger tips.

## Part 4: Guided Example: Detect Outliers with the 1.5 × IQR Rule

The box plot *showed* some outliers. Now we will **calculate** exactly which bills they are, the same way the box plot does.

$$\text{Lower fence} = Q_1 - 1.5 \times IQR \qquad \text{Upper fence} = Q_3 + 1.5 \times IQR$$

### Step 1: Compute the quartiles and IQR

In [ ]:
q1 = tips["total_bill"].quantile(0.25)
q3 = tips["total_bill"].quantile(0.75)
iqr = q3 - q1
print(f"Q1 = {q1:.2f}, Q3 = {q3:.2f}, IQR = {iqr:.2f}")

### Step 2: Compute the fences

In [ ]:
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
print(f"Lower fence = {lower_fence:.2f}   Upper fence = {upper_fence:.2f}")

A bill can't be negative, so nothing will fall below the lower fence. All the action is at the top.

### Step 3: Filter the outliers

In [ ]:
outliers = tips[(tips["total_bill"] < lower_fence) | (tips["total_bill"] > upper_fence)]

print("Number of outlier bills:", len(outliers))
outliers.sort_values("total_bill", ascending=False)

### Step 4: Visualize the fences on the histogram

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(tips["total_bill"], bins=20)
plt.axvline(upper_fence, color="red", linestyle="--", label=f"Upper fence ${upper_fence:.2f}")
plt.title("Bills Beyond the Upper Fence Are Flagged as Outliers")
plt.legend()
plt.show()

### Step 5: Decide what an outlier *means*
Outliers are **flags, not verdicts**. Look at the rows: most are **dinner** bills for parties of **3 or more**. These are real, legitimate big tables, not data-entry errors. **Keep them**, but remember they pull the mean upward, so report the **median** as the "typical bill."

### Step 6: Turn it into a reusable function

In [ ]:
def iqr_outliers(series):
    """Return the values in a numeric Series that fall outside the 1.5 x IQR fences."""
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return series[(series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)]

print("Outlier tips:", sorted(iqr_outliers(tips["tip"]).round(2).tolist()))

## Part 5: ✏️ Your Turn: Palmer Penguins EDA Sprint

The **penguins** dataset measures 344 penguins of three species on three Antarctic islands. Complete the four sprint tasks.

In [ ]:
penguins = sns.load_dataset("penguins")
penguins.head()

**Task 1: Check missing values and data types.** *Hint:* `.info()` and `.isna().sum()`

In [ ]:
# Your code here

<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
penguins.info()
print(penguins.isna().sum())
```

</details>

**Task 2: Univariate statistics for `flipper_length_mm`:** mean, median and IQR. *Hint:* reuse the code from Part 2.

In [ ]:
flipper = penguins["flipper_length_mm"]
# Your code here: print the mean, median, and IQR

<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
flipper = penguins["flipper_length_mm"]
print("Mean:  ", round(flipper.mean(), 1))
print("Median:", flipper.median())
print("IQR:   ", flipper.quantile(0.75) - flipper.quantile(0.25))
```

</details>

**Task 3: Scatter plot** of `flipper_length_mm` (x) vs. `body_mass_g` (y), colored by `species`. *Hint:* `sns.scatterplot(data=..., x=..., y=..., hue=...)`

In [ ]:
plt.figure(figsize=(7, 5))
# Your code here

plt.show()

<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
plt.figure(figsize=(7, 5))
sns.scatterplot(data=penguins, x="flipper_length_mm", y="body_mass_g", hue="species")
plt.title("Flipper Length vs. Body Mass by Species")
plt.show()
```

</details>

**Task 4: Write a 2-sentence business summary of your findings.** Double-click this cell to edit it.

*Your summary:* ...

<details>
<summary><b>💡 Reveal a sample answer</b></summary>

*Penguins with longer flippers are consistently heavier: the relationship is strong and positive across all three species. Gentoo penguins form a clearly separate, larger cluster, so species should be accounted for in any analysis of body size. Note that 2 rows are missing measurements and 11 are missing `sex`.*

</details>

## Part 6: 🧩 Guided Challenge: Penguins Deep Dive

Go beyond the sprint with three bivariate techniques from tonight's lecture.

**Step 1: Category vs. category: a crosstab.** Which species live on which islands? `pd.crosstab(row_variable, column_variable)` counts every combination.

In [ ]:
pd.crosstab(penguins["___"], penguins["___"], margins=True)

<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
pd.crosstab(penguins["species"], penguins["island"], margins=True)
```

</details>

**Step 2: Category vs. numeric: a grouped box plot.** Compare `body_mass_g` across `species`, split by `sex` with `hue`. *Which species shows the biggest male/female difference?*

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=penguins, x="___", y="___", hue="___")
plt.title("Body Mass by Species and Sex")
plt.show()

<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
plt.figure(figsize=(8, 5))
sns.boxplot(data=penguins, x="species", y="body_mass_g", hue="sex")
plt.title("Body Mass by Species and Sex")
plt.show()
```

</details>

**Step 3: Outliers *within* each group.** A 6,000 g penguin is normal for a Gentoo but would be extreme for an Adelie. Use your `iqr_outliers()` function **per species** by looping through the groups.

In [ ]:
for species_name, group in penguins.groupby("species"):
    flagged = iqr_outliers(group["___"].dropna())
    print(f"{species_name}: {len(flagged)} outlier(s) -> {flagged.tolist()}")

<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
for species_name, group in penguins.groupby("species"):
    flagged = iqr_outliers(group["body_mass_g"].dropna())
    print(f"{species_name}: {len(flagged)} outlier(s) -> {flagged.tolist()}")
```

</details>

**Step 4: Correlation heatmap for the penguins' numeric columns.** Which pair has the strongest correlation? Is any correlation negative?

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(penguins.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", vmin=___, vmax=___)
plt.title("Penguin Measurements: Correlation Heatmap")
plt.show()

<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
plt.figure(figsize=(6, 5))
sns.heatmap(penguins.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Penguin Measurements: Correlation Heatmap")
plt.show()
```

</details>

<details>
<summary><b>💡 Reveal what to look for</b></summary>

- **Crosstab:** Gentoo live only on Biscoe and Chinstrap only on Dream, while Adelie live on all three islands.
- **Box plot:** males are heavier in every species, and the gap is largest for **Gentoo**.
- **Outliers by group:** the overall body-mass column has **no** outliers, but within Chinstrap two birds (2,700 g and 4,800 g) are flagged. They only stand out when compared with their own species, which is why group-level checks matter.
- **Heatmap:** flipper length & body mass have the strongest correlation (~0.87). Bill depth is *negatively* correlated with flipper length and body mass overall, yet *positive within each species*. This is a famous example of **Simpson's paradox**, and why you should always check group-level patterns!

</details>

---
## ✅ Lab Checklist
- [ ] Computed mean, median, mode, variance, std and IQR
- [ ] Built a histogram, box plot, scatter plot and correlation heatmap
- [ ] Calculated 1.5 × IQR fences and flagged outliers
- [ ] Completed the penguins EDA sprint and wrote a business summary
- [ ] Used a crosstab, grouped box plot and per-group outlier checks

## 💾 Save Your Work to GitHub
1. In Colab, click **File → Save a copy in GitHub**.
2. Choose your course/Capstone repository.
3. Use this commit message: `Session 7: Completed EDA in Python Lab`

**Coming up next:** Session 8: Descriptive statistics in depth + Capstone Checkpoint 1 (bring your cleaned Capstone dataset!).